# Understanding US Flight Delays: What Makes Flights Late?


## An Analysis of 2025 Airline On-Time Performance Data

## by Luis Feliu


## Investigation Overview

This presentation communicates the key findings from an exploratory analysis of **7 million US domestic flights in 2025**. The data comes from the Bureau of Transportation Statistics (BTS) Reporting Carrier On-Time Performance database.

**Five key insights:**

1. **Delays compound over the flight** — late departures tend to arrive even later
2. **Airline operations drive delays** — LateAircraftDelay and CarrierDelay dominate
3. **Friday is the worst day** — end-of-week travel rush and operational fatigue
4. **Seasonal bimodality** — winter weather and summer thunderstorms both cause delays
5. **Hub congestion matters** — major hub airports show consistently higher delays

These insights reveal that delays are primarily driven by airline operational efficiency, schedule timing, seasonality, and hub congestion.


## Dataset Overview

**Source:** Bureau of Transportation Statistics (BTS) Reporting Carrier On-Time Performance Data  
**Coverage:** US domestic scheduled flights, 2025  
**Size:** ~7 million flights across 12 months  
**Project data location:** monthly source ZIP files in `data/raw/`

**Key variables:**
- **DepDelay / ArrDelay:** Departure and arrival delays in minutes (positive = late)
- **Delay causes:** CarrierDelay, WeatherDelay, NASDelay, SecurityDelay, LateAircraftDelay
- **Flight status:** Cancelled, Diverted
- **Route:** Origin, Dest, Distance
- **Time:** Month, DayOfWeek, departure time block

**Important note:** In this dataset, `NaN` in delay columns means the flight was on time (0 delay), not missing data. This is a critical distinction for correct analysis.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from zipfile import ZipFile

# Plotting style — polished for presentation
sns.set_style('whitegrid')
sns.set_context('notebook', font_scale=1.0)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 12

# Suppress warnings
import warnings
warnings.simplefilter('ignore')


In [ ]:
cols = ['Year','Month','DayofMonth','DayOfWeek','FlightDate',
    'IATA_CODE_Reporting_Airline','Origin','OriginCityName','OriginState',
    'Dest','DestCityName','DestState',
    'CRSDepTime','DepTime','DepDelay','DepDelayMinutes','DepDel15',
    'DepartureDelayGroups','DepTimeBlk',
    'CRSArrTime','ArrTime','ArrDelay','ArrDelayMinutes','ArrDel15',
    'ArrivalDelayGroups','ArrTimeBlk',
    'Cancelled','CancellationCode','Diverted',
    'ActualElapsedTime','AirTime','Flights','Distance','DistanceGroup',
    'CarrierDelay','WeatherDelay','NASDelay','SecurityDelay','LateAircraftDelay']

DATA_DIR = Path('data/raw')
PROCESSED_DATA = Path('data/processed/combined_2025.csv')
ZIP_PATTERN = 'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_*.zip'


def month_number(path):
    return int(path.stem.rsplit('_', 1)[-1])


def read_monthly_zip(zip_path, columns):
    with ZipFile(zip_path) as archive:
        csv_name = next(name for name in archive.namelist() if name.lower().endswith('.csv'))
        with archive.open(csv_name) as csv_file:
            return pd.read_csv(csv_file, usecols=columns, encoding='latin-1', low_memory=False)


if PROCESSED_DATA.exists():
    df = pd.read_csv(PROCESSED_DATA, usecols=cols, encoding='latin-1', low_memory=False)
else:
    zip_files = sorted(DATA_DIR.glob(ZIP_PATTERN), key=month_number)
    if len(zip_files) != 12:
        raise FileNotFoundError(
            'Expected 12 monthly 2025 BTS ZIP files in data/raw/. '
            'Copy them from the local resources/ folder or download them from BTS.'
        )
    df = pd.concat((read_monthly_zip(path, cols) for path in zip_files), ignore_index=True)

# Convert NaN delays to 0 (on-time flights)
delay_cols = ['DepDelay', 'ArrDelay', 'CarrierDelay', 'WeatherDelay',
              'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
              'DepDelayMinutes', 'ArrDelayMinutes', 'ActualElapsedTime', 'AirTime']
for col in delay_cols:
    df[col] = df[col].fillna(0)

# Derived features
df['DelayDelta'] = df['ArrDelay'] - df['DepDelay']
df['IsDelayed'] = df['ArrDelay'] > 15

# Top airlines
top_airlines = df['IATA_CODE_Reporting_Airline'].value_counts().head(10).index.tolist()
df['AirlineGroup'] = df['IATA_CODE_Reporting_Airline'].apply(
    lambda x: x if x in top_airlines else 'Other'
)

# Active flights (non-cancelled)
df_active = df[~df['Cancelled'].astype(bool)].copy()

# Day names
day_names = {1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat', 7: 'Sun'}
df_active['DayName'] = df_active['DayOfWeek'].map(day_names)

print(f"Dataset: {len(df):,} flights, {len(df_active):,} active (non-cancelled)")
print(f"Mean DepDelay: {df_active['DepDelay'].mean():.1f} min, Mean ArrDelay: {df_active['ArrDelay'].mean():.1f} min")
print(f"Mean DelayDelta: {df_active['DelayDelta'].mean():.1f} min (positive = delays compound)")


The cells above (imports and data loading) are hidden from the presentation. They run silently in the background.


## Insight 1: Delays Compound Over the Flight

**Question:** Do late departures lead to late arrivals? Do delays get worse or better during the flight?

**Answer:** Yes — delays consistently compound. The median delay delta (ArrDelay − DepDelay) is **positive**, meaning flights tend to arrive later relative to schedule than they departed. Late departures are particularly prone to worsening en route.

The visualization below shows the relationship between departure and arrival delays. The dashed diagonal line (y = x) represents no change. Most flights lie **above** this line, confirming that delays accumulate over the course of a flight.


In [ ]:
# Insight 1: DepDelay vs ArrDelay scatter
sample = df_active.sample(n=100000, random_state=42)

fig, ax = plt.subplots(figsize=(12, 7))
hb = ax.hexbin(sample['DepDelay'], sample['ArrDelay'], gridsize=40, cmap='YlOrRd', mincnt=1)
ax.set_title('Departure Delay vs Arrival Delay: Delays Compound Over the Flight', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Departure Delay (minutes)', fontsize=13)
ax.set_ylabel('Arrival Delay (minutes)', fontsize=13)
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.plot([0, 100], [0, 100], 'k--', alpha=0.4, linewidth=2, label='No change (y = x)')
ax.legend(fontsize=11, loc='upper left')
plt.colorbar(hb, label='Flight Count', pad=0.02, fraction=0.046)
ax.text(50, 85, 'Most flights above the line → delays compound', fontsize=11, ha='center',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='wheat', alpha=0.8))
plt.tight_layout()


The hexbin plot reveals a strong positive correlation between departure and arrival delays. The concentration of points above the y=x diagonal line is the key visual evidence: flights that depart late tend to arrive even later. This compounding effect is most pronounced for flights with departure delays exceeding 30 minutes.


## Insight 2: Airline Operations Drive Delays

**Question:** What are the primary causes of flight delays? Who has the most control over on-time performance?

**Answer:** Airline-controlled factors dominate. **LateAircraftDelay** (caused by late-arriving aircraft cascading into subsequent departures) and **CarrierDelay** (crew scheduling, maintenance, operational issues) are by far the largest contributors. Weather and security delays are relatively minor.

This means airlines have the most power to improve on-time performance — through better aircraft turnaround, crew management, and operational planning.


In [ ]:
# Insight 2: Mean delay by cause
delay_causes = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
cause_labels = ['Carrier\n(Airline Ops)', 'Weather', 'NAS\n(Air Traffic)', 'Security', 'LateAircraft\n(Cascading)']
cause_means = df_active[delay_causes].mean()

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336', '#9C27B0']
bars = ax.bar(cause_labels, cause_means.values, color=colors, edgecolor='white', linewidth=1.5, width=0.6)
ax.set_title('Mean Delay by Cause: Airline Operations Dominate', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Delay Cause', fontsize=13)
ax.set_ylabel('Mean Delay (minutes)', fontsize=13)
for bar, val in zip(bars, cause_means.values):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylim(0, max(cause_means.values) * 1.15)
plt.tight_layout()


The bar chart clearly shows that LateAircraftDelay and CarrierDelay dwarf all other causes. LateAircraftDelay reflects the cascading nature of airline operations — a late arrival at one airport delays the aircraft's next departure. CarrierDelay covers crew scheduling, maintenance, and other airline-controlled factors. Together, these two categories account for the vast majority of delay minutes, giving airlines the clearest path to improvement.


## Insight 3: Friday Is the Worst Day

**Question:** Are certain days of the week associated with more delays?

**Answer:** Yes — **Friday** has the highest median arrival delays, followed by Monday. Mid-week (Tuesday through Thursday) offers the best on-time performance. This pattern reflects two forces: the end-of-week travel rush (more flights, more congestion) and accumulated operational fatigue from the week's disruptions.

For travelers seeking the best chance of on-time arrival, Tuesday through Thursday are the optimal days.


In [ ]:
# Insight 3: ArrDelay by day of week
fig, ax = plt.subplots(figsize=(10, 6))
day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
palette = ['#1f77b4', '#1f77b4', '#1f77b4', '#1f77b4', '#d62728', '#1f77b4', '#1f77b4']
sns.boxplot(data=df_active, x='DayName', y='ArrDelay', ax=ax, palette=palette, order=day_order, fliersize=2)
ax.set_title('Arrival Delay by Day of Week: Friday Is the Worst', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Day of Week', fontsize=13)
ax.set_ylabel('Arrival Delay (minutes)', fontsize=13)
ax.axhline(df_active['ArrDelay'].median(), color='red', linestyle='--', alpha=0.5, label=f'Overall median: {df_active["ArrDelay"].median():.1f} min')
ax.legend(fontsize=10)
plt.tight_layout()


The box plot shows Friday with the highest median delay and the widest interquartile range, indicating both worse typical performance and more variability. Monday is also elevated, likely due to weekend maintenance backlogs. The red dashed line shows the overall median — Friday's median is clearly above it, while mid-week days cluster near or below it. Saturday shows lower median but higher variability, reflecting fewer flights and more diverse traffic patterns.


## Insight 4: Seasonal Bimodality

**Question:** Are there seasonal patterns in flight delays?

**Answer:** Yes — delays follow a **bimodal pattern** with two peaks: one in winter (December–February) driven by weather (snow, ice, fog), and another in summer (June–August) driven by thunderstorms and peak travel volume. Spring (March–May) and fall (September–November) offer the best on-time performance.

This bimodality suggests two distinct delay mechanisms at work, each dominant in different seasons.


In [ ]:
# Insight 4: ArrDelay by month
fig, ax = plt.subplots(figsize=(12, 6))
month_palette = ['#87CEEB', '#87CEEB', '#90EE90', '#90EE90', '#90EE90', '#FFA07A', '#FFA07A', '#FFA07A', '#90EE90', '#90EE90', '#87CEEB', '#87CEEB']
sns.boxplot(data=df_active, x='Month', y='ArrDelay', ax=ax, palette=month_palette)
ax.set_title('Arrival Delay by Month: Winter and Summer Peaks', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Month', fontsize=13)
ax.set_ylabel('Arrival Delay (minutes)', fontsize=13)
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
ax.set_xticklabels(month_labels)
ax.axhline(df_active['ArrDelay'].median(), color='red', linestyle='--', alpha=0.5, label=f'Overall median: {df_active["ArrDelay"].median():.1f} min')
ax.legend(fontsize=10)
plt.tight_layout()


The box plot reveals a clear bimodal pattern. Winter months (Jan, Feb, Dec) show elevated delays due to weather disruptions. Summer months (Jun, Jul, Aug) also show higher delays, driven by thunderstorm season and peak travel volume. The spring (Mar–May) and fall (Sep–Nov) months form a trough with the lowest median delays. This bimodality is important for both travelers (best months to fly) and airlines (seasonal staffing and scheduling adjustments).


## Insight 5: Hub Congestion Drives Delays

**Question:** Do some airports consistently delay departures? Do some hubs recover en route while others worsen?

**Answer:** States with major hub airports (New York, Florida, Illinois) show consistently higher departure delays. The gap between departure and arrival delays varies by state — some hubs recover en route (ArrDelay < DepDelay), while others worsen. This reveals whether delay problems are origin-focused (airport/airline operations) or route-focused (en route conditions).

Hub airports with large DepDelay but smaller ArrDelay gaps tend to have better en-route recovery capabilities.


In [ ]:
# Insight 5: DepDelay vs ArrDelay by origin state (top 10)
state_delays = df_active.groupby('OriginState').agg({
    'DepDelay': 'mean',
    'ArrDelay': 'mean'
}).sort_values('DepDelay', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(state_delays))
width = 0.35
bars1 = ax.bar(x - width/2, state_delays['DepDelay'], width, label='Mean DepDelay', color='steelblue', edgecolor='white')
bars2 = ax.bar(x + width/2, state_delays['ArrDelay'], width, label='Mean ArrDelay', color='coral', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(state_delays.index, rotation=45, ha='right')
ax.set_title('Departure vs Arrival Delay by Origin State: Hub Congestion', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Origin State', fontsize=13)
ax.set_ylabel('Mean Delay (minutes)', fontsize=13)
ax.legend(fontsize=11)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()


The clustered bar chart compares mean departure and arrival delays by origin state. States with major hubs (NY, FL, IL, CA, TX) show the highest departure delays. The gap between the blue (DepDelay) and coral (ArrDelay) bars tells an important story: when the bars are close together, the hub's delays tend to persist en route. When ArrDelay is notably lower than DepDelay, the hub has better en-route recovery. This distinction helps identify which hubs need operational improvements versus which need route-level solutions.


## Summary & Key Takeaways

**Five insights from 7 million flights:**

1. **Delays compound** — late departures tend to arrive even later
2. **Airlines control the most** — LateAircraftDelay and CarrierDelay dominate
3. **Friday is worst** — end-of-week rush and operational fatigue
4. **Bimodal seasons** — winter weather and summer storms both cause delays
5. **Hubs drive congestion** — major airports show consistently higher delays

**Bottom line:** Delays are primarily driven by airline operational efficiency and hub congestion, with timing and seasonality shaping when those delays become most severe. Airlines that invest in operational improvements — faster aircraft turnaround, better crew scheduling, and stronger hub management — have the greatest potential to improve on-time performance.

**For travelers:** Fly mid-week, choose airlines with strong operational track records, and be aware that Friday and winter months carry higher delay risk.


## Export

To generate the HTML slide deck:

```bash
jupyter nbconvert Part_II_explanatory.ipynb --to slides --no-input --no-prompt
```
